# Урок 11 - Протокол агент към агент (A2A)


## Настройка


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Какво е протоколът A2A?

Протоколът **Agent-to-Agent (A2A)** е отворен стандарт, който позволява на AI агенти да комуникират,
да се откриват взаимно и да си сътрудничат — дори когато са изградени на различни платформи или са хоствани
от различни услуги.

Основни понятия:

- **Откриване** – Агенти публикуват *Agent Card*, която описва техните възможности, улеснявайки
  други агенти (или оркестратор) да намерят правилния специалист за дадена задача.
- **Обмен на съобщения** – Агенти обменят структурирани съобщения чрез общ протокол, така че
  заявка от един агент да може да бъде разбрана и изпълнена от друг, независимо от вътрешната му
  реализация.
- **Жизнен цикъл на задачата** – A2A дефинира състояния като *подадена*, *в процес на работа*, *завършена* и
  *неуспешна*, давайки на оркестратора пълна видимост върху прогреса на делегираната задача.

В този урок симулираме колаборация в стил A2A, като свързваме три специализирани туристически агенти
в работен процес, където всеки агент допринася с експертизата си и предава резултатите на следващия.


## Създаване на специализирани туристически агенти


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Многоагентно сътрудничество чрез работен процес

Свързваме трите агента в последователен работен процес, който отразява предаването на съобщения A2A:

1. **CurrencyExchangeAgent** получава заявката на потребителя и дава указания за валутата.
2. **ActivityPlannerAgent** получава обогатения контекст и добавя препоръки за дейности.
3. **TravelManagerAgent** синтезира двата входа в окончателен пътеводител.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Разбиране на A2A в продукция

В производствена среда протоколът A2A отключва мощни сценарии за взаимодействие между услуги:

| Възможност | Описание |
|---|---|
| **Взаимодействие между различни рамки** | Агент, създаден с една рамка, може да делегира задачи на агент, създаден с която и да е друга рамка, съвместима с A2A, което позволява истинска съвместимост между организации. |
| **Граници на услугите** | Агентите могат да съществуват в отделни микросървиси, облачни региони или дори различни организации, като все пак работят заедно безпроблемно. |
| **Динамично откриване** | Оркестратор може да прави заявки към регистър на Agent Card по време на изпълнение, за да намери най-подходящия специалист за дадена подзадача. |
| **Поточно предаване и push известия** | A2A поддържа Server-Sent Events (SSE) за актуализации на прогреса в реално време и push известия за дълго изпълняващи се задачи. |

Работният процес, който изградихме по-горе, е опростена, вътрешнопроцесна версия на този модел. В реално
внедряване всеки агент би експонирал HTTP крайна точка, би публикувал Agent Card и би комуникирал
чрез протокола A2A JSON-RPC.


## Summary

In this lesson you learned:

1. **What the A2A protocol is** — an open standard for agent-to-agent discovery, messaging,
   and task management.
2. **How to create specialized agents** — a Currency Exchange agent, an Activity Planner agent,
   and a Travel Manager orchestrator.
3. **How to wire agents into a workflow** — using `WorkflowBuilder` to model sequential
   message passing between agents.
4. **How A2A works in production** — enabling cross-framework, cross-service collaboration
   with dynamic discovery and streaming updates.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от отговорност**:
Този документ е преведен с помощта на AI преводачески услуга [Co-op Translator](https://github.com/Azure/co-op-translator). Въпреки че се стремим към точност, моля имайте предвид, че автоматизираните преводи могат да съдържат грешки или неточности. Оригиналният документ на неговия роден език трябва да се счита за авторитетен източник. За критична информация се препоръчва професионален човешки превод. Ние не носим отговорност за каквито и да е недоразумения или неправилни тълкувания, произтичащи от използването на този превод.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
